# MMIA 6013 · Taller 01

# Parte 2.a — Parámetros expuestos por cada servicio



In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import requests
import pandas as pd
from datetime import date

from openai import OpenAI

from src.config import (
    OPENAI_API_KEY,
    OPENAI_ECONOMIC_MODEL,
    OPENAI_REASONING_MODEL,
    validate_environment
)

validate_environment()

client = OpenAI(api_key=OPENAI_API_KEY)

TEST_DATE = date.today().isoformat()

In [3]:
def test_openai_parameter(model_name, parameter, value):
    """
    Prueba un parámetro sobre un modelo de OpenAI.
    Retorna el resultado y el mensaje de error literal si existe.
    """
    payload = {
        "model": model_name,
        "input": "Respond only with the word: OK"
    }

    payload[parameter] = value

    try:
        client.responses.create(**payload)

        return {
            "accepted": True,
            "observation": "Accepted"
        }

    except Exception as e:

        return {
            "accepted": False,
            "observation": str(e)
        }

In [4]:
def test_qwen_parameter(parameter, value):
    """
    Prueba un parámetro sobre Qwen ejecutado en Ollama.
    """

    payload = {
        "model": "qwen3:1.7b",
        "prompt": "Respond only with the word: OK",
        "stream": False
    }

    payload[parameter] = value

    try:

        response = requests.post(
            "http://localhost:11434/api/generate",
            json=payload,
            timeout=60
        )

        response.raise_for_status()

        return {
            "accepted": True,
            "observation": "Accepted"
        }

    except Exception as e:

        return {
            "accepted": False,
            "observation": str(e)
        }

In [7]:
PARAMETERS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 20
}

MODELS = [
    ("gpt-4o-mini", OPENAI_ECONOMIC_MODEL),
    ("gpt-5.6", OPENAI_REASONING_MODEL),
    ("qwen3:1.7b", "qwen3:1.7b")
]

def build_parameter_matrix():

    rows = []

    for display_name, model_id in MODELS:

        for parameter, value in PARAMETERS.items():

            if display_name == "qwen3:1.7b":
                result = test_qwen_parameter(parameter, value)
            else:
                result = test_openai_parameter(
                    model_id,
                    parameter,
                    value
                )

            rows.append({
                "date": TEST_DATE,
                "model": display_name,
                "parameter": parameter,
                "value": value,
                "parameter_declared": "Yes" if result["accepted"] else "No",
                "observation": result["observation"]
            })

    return pd.DataFrame(rows)

In [8]:
parameter_matrix = build_parameter_matrix()

parameter_matrix

,date,model,parameter,value,parameter_declared,observation
0,2026-09-20,gpt-4o-mini,temperature,0.7,Yes,Accepted
1,2026-09-20,gpt-4o-mini,top_p,0.9,Yes,Accepted
2,2026-09-20,gpt-4o-mini,top_k,20.0,No,Responses.create() got an unexpected keyword a...
3,2026-09-20,gpt-5.6,temperature,0.7,No,"Error code: 400 - {'error': {'message': ""Unsup..."
4,2026-09-20,gpt-5.6,top_p,0.9,No,"Error code: 400 - {'error': {'message': ""Unsup..."
5,2026-09-20,gpt-5.6,top_k,20.0,No,Responses.create() got an unexpected keyword a...
6,2026-09-20,qwen3:1.7b,temperature,0.7,Yes,Accepted
7,2026-09-20,qwen3:1.7b,top_p,0.9,Yes,Accepted
8,2026-09-20,qwen3:1.7b,top_k,20.0,Yes,Accepted


## Conclusiones — Parte 2.a

La matriz fue obtenida mediante llamadas reales realizadas el 2026-09-20, registrando tanto los parámetros aceptados como los mensajes de error devueltos por cada servicio.

Los resultados muestran que GPT-4o mini expone los parámetros `temperature` y `top_p`, pero rechaza `top_k`. En contraste, GPT-5.6 rechazó los tres parámetros evaluados (`temperature`, `top_p` y `top_k`) en el endpoint utilizado, devolviendo errores de validación. Finalmente, Qwen 3 1.7B ejecutado mediante Ollama aceptó los tres parámetros de decodificación.

Esto confirma experimentalmente que la disponibilidad de los controles de generación depende del servicio y del endpoint, por lo que la compatibilidad debe verificarse mediante pruebas y no asumirse por documentación o memoria.

# Parte 2.b — Rejilla Temperature × Top-p

Se evalúan todas las combinaciones de:

- Temperature ∈ {0.0, 0.3, 0.7, 1.0, 1.5}
- Top-p ∈ {0.5, 0.9, 1.0}

Cada combinación se ejecuta 5 veces sobre los mismos 10 casos.

In [9]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0, 1.5]
TOP_P_VALUES = [0.5, 0.9, 1.0]

RUNS = 5

In [35]:
import json
import time

def generate_qwen(prompt, temperature, top_p):

    start = time.perf_counter()

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:1.7b",
            "prompt": prompt,
            "temperature": temperature,
            "top_p": top_p,
            "stream": False
        },
        timeout=300
    ).json()

    latency = time.perf_counter() - start

    predictions = json.loads(response["response"])

    return {
        "predictions": predictions,
        "latency": latency,
        "input_tokens": response["prompt_eval_count"],
        "output_tokens": response["eval_count"]
    }

In [36]:
def build_batch_prompt(cases_df):

    tickets = "\n".join(
        f"{row.id}. {row.ticket}"
        for _, row in cases_df.iterrows()
    )

    return f"""
You are a support ticket classifier.

Classify ALL 10 tickets.

Valid categories:
- HR
- Finance
- Technical

Return ONLY a JSON array with exactly 10 objects.

Required format:

[
  {{"id":1,"category":"HR"}},
  {{"id":2,"category":"Finance"}},
  ...
  {{"id":10,"category":"HR"}}
]

Tickets:
{tickets}
"""

In [39]:
def benchmark_temperature_topp(cases):

    rows = []

    prompt = build_batch_prompt(cases)

    for temp in TEMPERATURES:

        for top_p in TOP_P_VALUES:

            for run in range(1, RUNS + 1):

                output = generate_qwen(
                    prompt,
                    temp,
                    top_p
                )

                pred_dict = {
                    item["id"]: item["category"]
                    for item in output["predictions"]
                }

                for _, row in cases.iterrows():

                    prediction = pred_dict.get(row.id, "INVALID")

                    rows.append({
                        "run": run,
                        "temperature": temp,
                        "top_p": top_p,
                        "case_id": row.id,
                        "expected": row.expected,
                        "prediction": prediction,
                        "correct": prediction == row.expected,
                        "latency": output["latency"],
                        "output_tokens": output["output_tokens"]
                    })

    return pd.DataFrame(rows)

In [40]:
cases = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "casos.csv"
)

grid_results = benchmark_temperature_topp(cases)

grid_results.head()

,run,temperature,top_p,case_id,expected,prediction,correct,latency,output_tokens
0,1,0.0,0.5,1,HR,Technical,False,10.547873,823
1,1,0.0,0.5,2,Finance,Finance,True,10.547873,823
2,1,0.0,0.5,3,Technical,Technical,True,10.547873,823
3,1,0.0,0.5,4,HR,Finance,False,10.547873,823
4,1,0.0,0.5,5,Finance,Technical,False,10.547873,823


In [42]:
grid_summary = (
    grid_results
    .groupby(["temperature", "top_p"])
    .agg(
        accuracy=("correct", "mean"),
        avg_latency=("latency", "mean"),
        avg_output_tokens=("output_tokens", "mean")
    )
    .reset_index()
)

grid_summary["accuracy"] = (
    grid_summary["accuracy"] * 100
).round(1)

grid_summary["avg_latency"] = (
    grid_summary["avg_latency"]
).round(2)

grid_summary["avg_output_tokens"] = (
    grid_summary["avg_output_tokens"]
).round(0).astype(int)

grid_summary

,temperature,top_p,accuracy,avg_latency,avg_output_tokens
0,0.0,0.5,76.0,10.11,788
1,0.0,0.9,86.0,9.53,749
2,0.0,1.0,84.0,10.00,795
3,0.3,0.5,82.0,9.80,769
4,0.3,0.9,80.0,11.59,933
5,0.3,1.0,78.0,9.69,744
6,0.7,0.5,78.0,10.08,777
7,0.7,0.9,82.0,12.14,963
8,0.7,1.0,82.0,9.58,721
9,1.0,0.5,84.0,10.85,837


In [43]:
grid_results.to_csv(
    PROJECT_ROOT / "data" / "processed" / "parte2b_raw.csv",
    index=False
)

grid_summary.to_csv(
    PROJECT_ROOT / "data" / "processed" / "parte2b_summary.csv",
    index=False
)

## Conclusiones — Parte 2.b

Se evaluó una rejilla de 15 combinaciones (5 temperaturas × 3 valores de top_p), realizando 5 corridas por combinación sobre los mismos 10 casos.

La mayor exactitud obtenida fue 86 % con temperature = 0.0 y top_p = 0.9, además de una latencia promedio de 9.53 s y 749 tokens de salida. Esto sugiere que una generación determinista con un núcleo moderado produjo las clasificaciones más consistentes para este conjunto de datos.

No se observó una relación lineal entre temperatura y exactitud. Algunas configuraciones con temperaturas intermedias (0.7 y 1.0) mantuvieron rendimientos cercanos al 82–84 %, mientras que temperaturas altas (1.5) tendieron a reducir la exactitud hasta 76–80 %.

La longitud de salida varió entre 721 y 963 tokens, debido a que Qwen 3 mantuvo activado su modo de razonamiento. En consecuencia, los tokens de salida y la latencia incluyen tanto la respuesta final como el proceso de razonamiento generado por el modelo.